# **MODULO 2: Fraude en reproducciones stream:**



**NOTAS**

* Pipeline:

stream logs

↓

feature engineering

↓

EDA

↓

anomaly detection

↓

fraud score

* Modelo  ML debe buscar patrones como: 1000 streams, desde 1 usuario, en 1 hora, duración promedio 5 segundos

# **DATA**

## Recopilación de datos en Last.fm API


### Endpoints & Pipeline API

| Endpoint             | Devuelve     |
| -------------------- | ---------------- |
| `artist.getInfo`     | info del artista |
| `track.getInfo`      | info del track   |

Pipeline API:
1. Llamas a endpoints
2. Controlas velocidad (throttling)
3. Evitas errores (retry + backoff)
4. Añades margen de seguridad
5. Combinas datos


# **EDA**

**Definición de fraude:** (1) Reproducciones artificiales; (2) Bots / farms; (3) Patrones no humanos.

**Análisis exploratorio**
1. **Comportamiento de usuarios:** (1) Tiempo entre streams; (2) Número de canciones únicas; (3) Repeticiones por track.

2. **Patrones sospechosos:** (1) Streams continuos 24/7; (2) Baja diversidad musical; (3) Intervalos constantes (automatización).
3. **Feature engineering (CLAVE):** time_between_streams,  sessions por día,  entropía de escucha,  ratio repeticiones / canciones únicas.

4. **Modelos de detección**

* 3 enfoques posibles: (1) Reglas heurísticas (baseline); (2) Anomaly Detection; (3) Isolation Forest

* One-Class SVM, Clustering, detectar grupos raros.

5. **Output útil:** (1) Score de fraude por usuario; (2) Clasificación: normal vs sospechoso; (3) Dashboard de comportamiento.


## **Módulo 2. DETECCIÓN DE ANOMALÍAS DE USUARIO EN REPRODUCCIONES STREAMS**
* Se usan los datos de Last.fm API también. --> Completar al decidir

* Objetivo: Analizar el comportamiento de los usuarios para detectar anomalias. A través de la informaticón de timestamps y time_between_streams de podría averiguar si esa cuenta ussuario es un humano o un bot.

* EDA: analisis outliers- usarios vs diferencia de tiempo

* ML: clusterung


In [7]:
#### Obtener el historial de un usuario para revisar anomalías
# api_key = '63e059c3c912a3f642daf2372484d183'
# username = ['listeners'] # duda como sacarlo¿?
# def check_user_activity(api_key, username):
#     url = f"https://ws.audioscrobbler.com/2.0/?method=user.getRecentTracks&user={username}&api_key={api_key}&format=json"
#     data = requests.get(url).json()
#     # Aquí podrías calcular la diferencia de tiempo entre tracks (feature engineering) [5]
#     return data['recenttracks']['track']
#### Obtener lista de usuarios a través de amigos del usuario 'rj'

In [8]:
import requests
import pandas as pd

API_KEY = "63e059c3c912a3f642daf2372484d183"

def get_user_friends(username):
    """Obtiene la lista de amigos de un usuario para expandir la base de datos."""
    url = f"https://ws.audioscrobbler.com/2.0/?method=user.getfriends&user={username}&api_key={API_KEY}&format=json"
    response = requests.get(url).json()
    if 'friends' in response:     # Extraemos solo los nombres de usuario de la lista de amigos
        return [friend['name'] for friend in response['friends']['user']]
    return []

def get_recent_tracks(username):
    """Obtiene el historial reciente para detectar patrones de fraude."""
    url = f"https://ws.audioscrobbler.com/2.0/?method=user.getrecenttracks&user={username}&api_key={API_KEY}&format=json"
    data = requests.get(url).json()
    if 'recenttracks' in data:
        return data['recenttracks']['track']
    return []

lista_usuarios = get_user_friends("rj") # Empezamos con 'rj' y buscamos a sus amigos
lista_usuarios[:5]
#### Recolectamos en un al ista los datos de escucha de esos nuevos usuarios
lista_total_escuchas = []
for user in lista_usuarios[:5]:  
    tracks = get_recent_tracks(user)
    for t in tracks:
        t['user_id'] = user  # Añadimos el ID de usuario para el análisis 
        lista_total_escuchas.append(t)


In [9]:

# Creamos el DataFrame para EDA 
df_escuchas = pd.DataFrame(lista_total_escuchas)


In [10]:
df_escuchas['artist'][0] # muestra de datos guardados en el diccionario de la columna artist


{'mbid': '', '#text': 'Fleetwood Mac'}

## Limpieza:

In [11]:
## En algunas filas se encuentra información guardada en diccionarios y solo queremos visualizar un dato.
df_escuchas['artist'] = df_escuchas['artist'].apply(lambda x: x['#text'] if isinstance(x, dict) else x)
df_escuchas['artist'] = df_escuchas['album'].apply(lambda x: x['#text'] if isinstance(x, dict) else x)
df_escuchas['date'] = df_escuchas['date'].apply(lambda x: x['#text'] if isinstance(x, dict) else x) # --> limpiar mejor la fecha, hacer oclumna separada para la hora de la escucha


## Limpieza de variables

In [12]:
df_escuchas.drop(['url','image'],axis=1)


,artist,streamable,mbid,album,name,date,user_id
0,Tango in the Night (2017 Remaster),0,,"{'mbid': '', '#text': 'Tango in the Night (201...",Everywhere - 2017 Remaster,"09 Apr 2026, 06:21",Babs_05
1,Tango in the Night (2017 Remaster),0,,"{'mbid': '', '#text': 'Tango in the Night (201...",Seven Wonders - 2017 Remaster,"09 Apr 2026, 06:17",Babs_05
2,Tango in the Night (2017 Remaster),0,,"{'mbid': '', '#text': 'Tango in the Night (201...",Big Love - 2017 Remaster,"09 Apr 2026, 06:13",Babs_05
3,Tango in the Night (2017 Remaster),0,,"{'mbid': '', '#text': 'Tango in the Night (201...","You and I, Pt. II - 2017 Remaster","09 Apr 2026, 06:10",Babs_05
4,Tango in the Night (2017 Remaster),0,,"{'mbid': '', '#text': 'Tango in the Night (201...",When I See You Again - 2017 Remaster,"09 Apr 2026, 06:07",Babs_05
...,...,...,...,...,...,...,...
245,Kosmopolis,0,,"{'mbid': '', '#text': 'Kosmopolis'}",Klunton,"06 Apr 2026, 20:05",Knapster01
246,Kosmopolis,0,,"{'mbid': '', '#text': 'Kosmopolis'}",Come With Me To Oviedo,"06 Apr 2026, 18:33",Knapster01
247,Kosmopolis,0,,"{'mbid': '', '#text': 'Kosmopolis'}",Stare miasto,"06 Apr 2026, 18:26",Knapster01
248,Kosmopolis,0,,"{'mbid': '', '#text': 'Kosmopolis'}",Warsaw Brew,"06 Apr 2026, 18:18",Knapster01


## Limpieza de datos

## Detección de Anomalías (Análisis de Timestamps)

* Para identificar reproducciones artificiales, el método clave en Last.fm es user.getRecentTracks. Este recurso devuelve una lista de canciones escuchadas recientemente con timestamps exactos, lo que permite calcular el intervalo de tiempo entre cada reproducción.
* DATA: 

· En tiempo real desde Freesound: Usando el método user.getRecentTracks para cualquier usuario activo en la plataforma.

· Histórico masivo: En el Taste Profile subset del Million Song Dataset (MSD), que contiene datos de escucha de usuarios recopilados específicamente para investigación comercial. -> [DUDA] 

* Código para Detección de Anomalías (Análisis de Timestamps). Este código calcula el time_between_streams y el repeat_count, que son las variables que propusiste para detectar patrones de bots.



**API REQUEST PARA LA INFORMACIÓN DEL USUARIO 'RJ':**

* El usuario 'rj' servirá se base para recolectar más data de otros usuarios. 

In [15]:
# API Request : DUDA: QUE está Pasando? porque error 6? que siginica?
ret = requests.get(f"https://ws.audioscrobbler.com/2.0/?method=user.getRecentTracks&user='rj'&api_key=63e059c3c912a3f642daf2372484d183&format=json")
ret.json()
def time_between_streams(API_KEY, username):
    # Petición clave para ver el historial con timestamps [4]
    url = f"https://ws.audioscrobbler.com/2.0/?method=user.getRecentTracks&user={username}&api_key={API_KEY}&format=json"
    response = requests.get(url).json()
    tracks = response['recenttracks']['track']
    
    # Creamos un DataFrame para procesar los patrones temporales [7]
    data = []
    for t in tracks:
        if 'date' in t: # Solo tracks que ya terminaron de sonar
            data.append({
                'track': t['name'],
                'timestamp': int(t['date']['uts']) # Unix timestamp
            })
    
    df = pd.DataFrame(data)
    
    # Feature Engineering para anomalías [7]:
    # 1. Calcular tiempo entre canciones (en segundos)
    df['time_between_streams'] = df['timestamp'].diff().abs()
    
    # 2. Detectar reproducciones consecutivas de la misma canción (repeat_count)
    df['is_repeat'] = df['track'] == df['track'].shift()
    
    return df


In [16]:
# Reutilizo la lista de usuarios iniciales de Last.fm API
API_KEY = "63e059c3c912a3f642daf2372484d183"
time_between_streams(API_KEY, 'rj')

lista_usuarios


['Babs_05',
 'franhale',
 'eartle',
 'massdosage',
 'Knapster01',
 'jonocole',
 'isaac',
 'lobsterclaw',
 'jajo',
 'mremond',
 'Orlenay',
 'schlagschnitzel',
 'Edouard',
 'naniel',
 'dunk',
 'RUPERT',
 'mxcl',
 'jwheare',
 'nancyvw',
 'underpangs',
 'p_wheel',
 'spietsch',
 'musicmobs',
 'Schrollum',
 'luke_10',
 'tgwizard',
 'pkeanecbs',
 'Roelven',
 'BecFrost',
 'gracehn001',
 'saulklein',
 'arrdis',
 'jarvis',
 'fascinated',
 'noirsette',
 'marquezmj',
 'julians',
 'david',
 'pellitero',
 'claoi',
 'grazziee',
 'clairewkyb',
 'lumberjack',
 'foreverautumn',
 'Yllona',
 'cakemix',
 'Jonty',
 'nova77LF',
 'muz',
 'mischa']

In [17]:
import pandas as pd

lista_usuarios = get_user_friends("rj")

lista_tbs = []

for user in lista_usuarios:
    print(user)
    try:
        df_tbs = time_between_streams(API_KEY, user)
        
        # opcional: añadir columna con el usuario
        df_tbs["user"] = user
        
        lista_tbs.append(df_tbs)
        
    except Exception as e:
        print(f"Error con {user}: {e}")

# Unir todos los dataframes
df_final = pd.concat(lista_tbs, ignore_index=True)

df_final

df_final['user'].nunique()


Babs_05
franhale
eartle
massdosage
Knapster01
jonocole
isaac
lobsterclaw
jajo
mremond
Orlenay
schlagschnitzel
Error con schlagschnitzel: 'recenttracks'
Edouard
naniel
dunk
RUPERT
mxcl
jwheare
nancyvw
underpangs
p_wheel
spietsch
musicmobs
Schrollum
Error con Schrollum: 'recenttracks'
luke_10
tgwizard
pkeanecbs
Roelven
BecFrost
gracehn001
saulklein
arrdis
jarvis
fascinated
noirsette
marquezmj
julians
david
Error con david: 'recenttracks'
pellitero
claoi
grazziee
clairewkyb
lumberjack
foreverautumn
Yllona
cakemix
Jonty
nova77LF
muz
mischa


47

* **Gráficos**

In [ ]:
# Hacer graficos de tbs comparar con actividad user (canciones: track, is_repeat)
# cambiar timestamp a formato fecha
#### Con API MSD

data_users_msd = 'http://millionsongdataset.com/sites/default/files/tasteprofile/taste_profile_usercat_120k.txt'
df_users_msd = pd.read_csv(data_users_msd,sep='\t',header=None,names=['user_id', 'song_id', 'play_count'])
import pandas as pd


In [ ]:
# Código ejemplo:
# Carga del subconjunto de datos de usuarios (Taste Profile)

def load_msd_user_data(file_path):
    # Cargamos el archivo (ejemplo con un archivo de texto tabulado)
    df_msd = pd.read_csv(file_path, sep='\t', names=['user_id', 'song_id', 'play_count'])
    
    # Feature Engineering para Detección de Fraude [3]:
    # Calculamos el 'repeat_ratio' que propusiste en tu documentación
    user_stats = df_msd.groupby('user_id')['play_count'].agg(['sum', 'max']).reset_index()
    user_stats['repeat_ratio'] = user_stats['max'] / user_stats['sum']
    
    # Identificamos posibles anomalías (por ejemplo, alguien que escucha una sola canción el 90% del tiempo)
    posibles_bots = user_stats[user_stats['repeat_ratio'] > 0.9]
    return posibles_bots
data_users_msd = 'http://millionsongdataset.com/sites/default/files/tasteprofile/taste_profile_usercat_120k.txt'
load_msd_user_data(data_users_msd)


,user_id,sum,max,repeat_ratio
